In [ ]:
# !pip install linearmodels # Install linearmodels package
# !pip install pydynpd # Install pydynpd package

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 21.2 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import warnings
import math
from scipy import stats
from linearmodels.panel import PooledOLS, PanelOLS, RandomEffects
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tsa.stattools import adfuller
import matplotlib.pyplot as plt # Added import statement
warnings.filterwarnings("ignore")

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
WARNA_UTAMA = "#1F4E78"
WARNA_AKSEN = "#C0392B"
WARNA_HIJAU = "#548235"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# =====================================================================
# TAHAP 0 -- TAMBALAN pydynpd (numpy 2.x + patch Hansen/AR) DAN
#            KONFIGURASI PATH
# =====================================================================
if not hasattr(np, "in1d"):
    np.in1d = np.isin
import pydynpd.specification_tests as _pydynpd_tests
from scipy import stats as _stats

def _hansen_overid_patched(W2_inv, N, zs, num_instru, num_indep):
    hansen_test = np.linalg.multi_dot([zs.transpose(), W2_inv, zs]) * (1.0 / N)
    df_ = num_instru - num_indep
    crit = _stats.chi2.ppf(q=0.95, df=df_)
    p_value = 1 - _stats.chi2.cdf(hansen_test, df_)
    return _pydynpd_tests.hansen_test_info(
        float(np.asarray(hansen_test).item()), df_,
        float(np.asarray(p_value).item()), crit)

def _AR_test_patched(model, zs_list, step, m):
    N = model.N
    z_height = int(model.z_list.shape[0] / N)
    current_step = model.step_results[step - 1]
    ori_residual = current_step.residual
    r0_height = int(ori_residual.shape[0] / N)
    M_XZ_W = current_step._M_XZ_W
    vcov = current_step.vcov
    diff_x, diff_r = _pydynpd_tests.AR_get_diff_XR(model, current_step.beta, ori_residual, r0_height)
    r_height = int(diff_r.shape[0] / N)
    x_height = int(diff_x.shape[0] / N)
    AR_list = []
    temp = np.zeros((r_height * N, 1), np.float64)
    _pydynpd_tests.lag(diff_r, temp, N, 1, 0)
    for j in range(1, m + 1):
        for i in range(N):
            r_i = diff_r[(r_height * i):(r_height * i + r_height), 0:1]
            r_t_i = r_i.transpose()
            lag_res = np.ndarray((r_height, 1), dtype=np.float64)
            _pydynpd_tests.lag(r_i, lag_res, 1, j, 0)
            lag_res[np.isnan(lag_res)] = 0
            lag_res_t = lag_res.transpose()
            x = diff_x[(x_height * i):(x_height * i + x_height), :]
            d0_temp = lag_res_t @ r_i
            d1_temp = d0_temp @ r_t_i @ lag_res
            EX_temp = lag_res_t @ x
            zs = zs_list[(z_height * i):(z_height * i + z_height), :]
            temp3_temp = zs @ d0_temp.transpose()
            if i == 0:
                d0, d1, EX, temp3 = d0_temp, d1_temp, EX_temp, temp3_temp
            else:
                d0 += d0_temp; d1 += d1_temp; EX += EX_temp; temp3 += temp3_temp
        d2 = (-2) * np.linalg.multi_dot([EX, M_XZ_W, temp3])
        d3 = np.linalg.multi_dot([EX, vcov, EX.transpose()])
        d0_s = float(np.asarray(d0).item())
        denom = float(np.asarray(d1 + d2 + d3).item())
        AR_temp = d0_s / math.sqrt(denom)
        P_value = _stats.norm.sf(abs(AR_temp)) * 2
        AR_list.append(_pydynpd_tests.AR_test_info(j, AR_temp, P_value))
    return AR_list

_pydynpd_tests.hansen_overid = _hansen_overid_patched
_pydynpd_tests.AR_test = _AR_test_patched
from pydynpd import regression
regression.tests.hansen_overid = _hansen_overid_patched
regression.tests.AR_test = _AR_test_patched
# --- akhir tambalan wajib ---

FOLDER_OUTPUT = "/content/drive/MyDrive/DatabaseSkripsi/Outputs"
FOLDER_SUMBER = "/content/drive/MyDrive/DatabaseSkripsi/SumberData"
ENTITAS_KHUSUS = "KALIMANTAN UTARA"   # T lebih pendek, baru terbentuk 2012


def bersihkan_angka_indo(kolom_series):
    bersih = (kolom_series.astype(str).str.strip()
              .str.replace(".", "", regex=False).str.replace(",", ".", regex=False))
    return pd.to_numeric(bersih, errors="coerce")

In [ ]:
# #####################################################################
# TAHAP I -- PERSIAPAN DATA PANEL
# #####################################################################
print(f"{'='*70}\nTAHAP I -- PERSIAPAN DATA PANEL\n{'='*70}")

pdrb_hijau = pd.read_csv(f"{FOLDER_OUTPUT}/master_panel_pdrb_hijau_2010_2025.csv")
kontrol = pd.read_csv(f"{FOLDER_OUTPUT}/fitur_tahunan_kontrol_gmm_ndvi_lst.csv")
pmtb = pd.read_csv(f"{FOLDER_SUMBER}/PMTB.csv").rename(
    columns={"Tahun": "year", "Provinsi": "PROVINSI", "PMTB": "PMTB_ADHK_Miliar_Rp"})
pmtb["PMTB_ADHK_Miliar_Rp"] = bersihkan_angka_indo(pmtb["PMTB_ADHK_Miliar_Rp"])
tenaga_kerja = pd.read_csv(f"{FOLDER_SUMBER}/Tenaga_Kerja.csv").rename(
    columns={"Tahun": "year", "Provinsi": "PROVINSI", "Bekerja": "Jumlah_Bekerja_Orang"})
tenaga_kerja["Jumlah_Bekerja_Orang"] = bersihkan_angka_indo(tenaga_kerja["Jumlah_Bekerja_Orang"])
ipm = pd.read_csv(f"{FOLDER_SUMBER}/IPM.csv").rename(columns={"Tahun": "year", "Provinsi": "PROVINSI"})
ipm["IPM"] = bersihkan_angka_indo(ipm["IPM"])
pdrb_kapita = pd.read_csv(f"{FOLDER_SUMBER}/PDRB_per_Kapita_ADHK.csv")
pdrb_kapita.columns = ["PROVINSI", "year", "pdrb_per_kapita_ribu_rp"]
pdrb_kapita["pdrb_per_kapita_ribu_rp"] = bersihkan_angka_indo(pdrb_kapita["pdrb_per_kapita_ribu_rp"])

kerangka = pdrb_hijau[["PROVINSI", "year"]].drop_duplicates()
makro = (kerangka.merge(pmtb, on=["PROVINSI", "year"], how="left")
                 .merge(tenaga_kerja, on=["PROVINSI", "year"], how="left")
                 .merge(ipm, on=["PROVINSI", "year"], how="left")
                 .merge(pdrb_kapita, on=["PROVINSI", "year"], how="left"))
df = pdrb_hijau.merge(makro, on=["PROVINSI", "year"], how="left").merge(kontrol, on=["PROVINSI", "year"], how="left")
df = df.sort_values(["PROVINSI", "year"]).reset_index(drop=True)

# I.2 -- Struktur keseimbangan panel
rekap_T = df.groupby("PROVINSI")["year"].count()
print("\n[I.2] Jumlah observasi per provinsi (harus SAMA kecuali Kalimantan Utara):")
print(rekap_T.sort_values())
print(f"Total observasi riil: {len(df)} (34 provinsi, TIDAK balanced by design)")

# I.3 -- Audit missing data per kolom sumber (SEBELUM transformasi)
print("\n[I.3] Jumlah NaN per kolom sumber:")
kolom_sumber = ["pdrb_hijau_utama", "PMTB_ADHK_Miliar_Rp", "Jumlah_Bekerja_Orang",
                 "IPM", "pdrb_per_kapita_ribu_rp", "ndvi_mean", "lst_mean"]
print(df[kolom_sumber].isna().sum())

# I.3b -- Cek celah DI TENGAH deret tahun
print("\n[I.3b] Cek kontinuitas tahun per provinsi (celah di tengah = risiko GMM):")
for kolom_cek in ["Jumlah_Bekerja_Orang", "IPM", "pdrb_per_kapita_ribu_rp"]:
    for prov, grup in df.groupby("PROVINSI"):
        tahun_ada = grup.loc[grup[kolom_cek].notna(), "year"].sort_values().tolist()
        if len(tahun_ada) < 2:
            continue
        celah_tengah = [t for i, t in enumerate(tahun_ada[1:], 1) if t - tahun_ada[i - 1] > 1]
        if celah_tengah:
            print(f"  [PERHATIAN] {prov} / {kolom_cek}: celah sebelum {celah_tengah}")

# I.4 -- Deteksi outlier
print("\n[I.4] Outlier per IQR (periksa manual -- valid secara fisik atau input error?):")
for col in ["pdrb_hijau_utama", "PMTB_ADHK_Miliar_Rp", "IPM", "pdrb_per_kapita_ribu_rp"]:
    s = df[col].dropna()
    q1, q3 = s.quantile([0.25, 0.75]); iqr = q3 - q1
    n_outlier = ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum()
    if n_outlier:
        print(f"  {col}: {n_outlier} observasi di luar pagar IQR")

# I.5 -- Transformasi (log level + growth = log first-difference)
df["ln_pdrb_hijau"] = np.log(df["pdrb_hijau_utama"])
df["ln_pmtb_per_pekerja"] = np.log(df["PMTB_ADHK_Miliar_Rp"] * 1e9 / df["Jumlah_Bekerja_Orang"])
df["ln_pdrb_kapita"] = np.log(df["pdrb_per_kapita_ribu_rp"])
df["growth_pdrb_hijau"] = df.groupby("PROVINSI")["ln_pdrb_hijau"].diff()
print(f"\n[I.5] growth_pdrb_hijau non-null: {df['growth_pdrb_hijau'].notna().sum()} "
      f"(harapan {len(df)}-34={len(df)-34}, tahun pertama tiap provinsi tak py pertumbuhan)")

# Ekspor panel
df.to_csv(f"{FOLDER_OUTPUT}/panel_final_gmm_tahap1.csv", index=False)
print(f"\n[I -- OUTPUT] Panel final disimpan -> panel_final_gmm_tahap1.csv "
      f"(dipakai ulang oleh skrip R, JANGAN re-derive manual di R).")

TAHAP I -- PERSIAPAN DATA PANEL

[I.2] Jumlah observasi per provinsi (harus SAMA kecuali Kalimantan Utara):
PROVINSI
ACEH                    16
BALI                    16
BANTEN                  16
BENGKULU                16
DI YOGYAKARTA           16
DKI JAKARTA             16
GORONTALO               16
JAMBI                   16
JAWA BARAT              16
JAWA TENGAH             16
JAWA TIMUR              16
KALIMANTAN BARAT        16
KALIMANTAN SELATAN      16
KALIMANTAN TENGAH       16
KALIMANTAN TIMUR        16
KALIMANTAN UTARA        16
KEP. BANGKA BELITUNG    16
KEP. RIAU               16
LAMPUNG                 16
MALUKU                  16
MALUKU UTARA            16
NUSA TENGGARA BARAT     16
NUSA TENGGARA TIMUR     16
PAPUA                   16
PAPUA BARAT             16
RIAU                    16
SULAWESI BARAT          16
SULAWESI SELATAN        16
SULAWESI TENGAH         16
SULAWESI TENGGARA       16
SULAWESI UTARA          16
SUMATERA BARAT          16
SUMATERA SELATAN   

In [ ]:
# #####################################################################
# TAHAP II -- STATISTIK DESKRIPTIF & EKSPLORASI
# #####################################################################
print(f"\n{'='*70}\nTAHAP II -- STATISTIK DESKRIPTIF\n{'='*70}")
KOLOM_INTI = ["growth_pdrb_hijau", "ln_pmtb_per_pekerja", "IPM", "ln_pdrb_kapita", "ndvi_mean", "lst_mean"]
print("\n[II.1] Statistik deskriptif (overall):")
print(df[KOLOM_INTI].describe().T)

# II.1b -- Dekomposisi overall/between/within
print("\n[II.1b] Dekomposisi overall/between/within:")
for col in KOLOM_INTI:
    s = df.set_index(["PROVINSI", "year"])[col]
    between = s.groupby(level=0).transform("mean")
    print(f"  {col:22s} sd_overall={s.std():.4f}  sd_between={s.groupby(level=0).mean().std():.4f}  "
          f"sd_within={(s - between + s.mean()).std():.4f}")

print("\n[II.2] Matriks korelasi antar regressor (tanda dini multikolinearitas):")
print(df[["ln_pmtb_per_pekerja", "IPM", "ln_pdrb_kapita", "ndvi_mean", "lst_mean"]].corr().round(3))

X_vif = df[["ln_pmtb_per_pekerja", "IPM", "ln_pdrb_kapita", "ndvi_mean", "lst_mean"]].dropna()
vif_data = pd.DataFrame({"variabel": X_vif.columns,
                          "VIF": [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]})
print("\n[II.2b] VIF (ambang perhatian: 10):")
print(vif_data.to_string(index=False))


TAHAP II -- STATISTIK DESKRIPTIF

[II.1] Statistik deskriptif (overall):
                     count       mean       std        min        25%  \
growth_pdrb_hijau    505.0   0.050906  0.089761  -0.549012   0.030187   
ln_pmtb_per_pekerja  539.0  16.917216  0.604099  15.732400  16.519756   
IPM                  541.0  70.284140  4.954906  54.450000  67.200000   
ln_pdrb_kapita       541.0  10.435821  0.591472   9.139573  10.067547   
ndvi_mean            544.0   0.770613  0.088180   0.330727   0.746609   
lst_mean             544.0  28.774991  2.027317  24.036167  27.605653   

                           50%        75%        max  
growth_pdrb_hijau     0.052196   0.066872   0.772563  
ln_pmtb_per_pekerja  16.775081  17.135978  18.915029  
IPM                  70.330000  73.630000  85.050000  
ln_pdrb_kapita       10.331894  10.616719  12.264123  
ndvi_mean             0.799693   0.822118   0.850600  
lst_mean             28.486044  29.773691  38.066642  

[II.1b] Dekomposisi overall/

In [ ]:
# #####################################################################
# TAHAP III -- UJI PRASYARAT DATA PANEL (UNIT ROOT & CROSS-SECTIONAL DEP.)
# #####################################################################
print(f"\n{'='*70}\nTAHAP III -- UJI PRASYARAT PANEL\n{'='*70}")


def fisher_maddala_wu(dframe, kolom_provinsi, kolom_nilai, maxlag=1):
    """Uji Fisher/Maddala-Wu (1999) -- kombinasi p-value ADF per unit.
    Dipertahankan dari uji_stasioneritas_formal.py (SUDAH BENAR, formula
    eksak, tidak bergantung tabel Monte Carlo spt LLC/IPS asli)."""
    p_values = []
    for _, grup in dframe.groupby(kolom_provinsi):
        seri = grup[kolom_nilai].dropna().values
        if len(seri) < 5:
            continue
        try:
            p_values.append(adfuller(seri, maxlag=maxlag, autolag=None)[1])
        except Exception:
            continue
    p_values = np.clip(np.array(p_values), 1e-10, 1 - 1e-10)
    N = len(p_values)
    P_stat = -2 * np.sum(np.log(p_values))
    p_gabungan = 1 - stats.chi2.cdf(P_stat, df=2 * N)
    return P_stat, 2 * N, p_gabungan, N


for label, kolom in [("LEVEL ln(PDRB Hijau)", "ln_pdrb_hijau"), ("PERTUMBUHAN Delta ln", "growth_pdrb_hijau")]:
    P_stat, df_chi2, p_val, N = fisher_maddala_wu(df, "PROVINSI", kolom)
    kesimpulan = "STASIONER" if p_val < 0.05 else "TIDAK STASIONER (dekat unit root)"
    print(f"[III.1] Fisher/Maddala-Wu -- {label}: P={P_stat:.4f} (df={df_chi2}, N={N}), "
          f"p={p_val:.6f} -> {kesimpulan}")

print(
    "\n[III -- CATATAN WAJIB] Fisher/Maddala-Wu SAH secara metodologis (Maddala & Wu, "
    "1999) dan justru LEBIH cocok utk panel unbalanced (Kalimantan Utara) drpd LLC "
    "yang butuh T seragam. TAPI penguji sering mengharapkan nama uji standar LLC/IPS "
    "-- kedua uji itu TIDAK tersedia matang di Python. WAJIB jalankan Tahap III di "
    "skrip R (levinLin, ips.test) sbg pelengkap/penguat, bukan pengganti.")

# III.2 -- Pesaran CD --
print("\n[III.2] Pesaran CD test (cross-sectional dependence) -- spesifikasi growth:")
kolom_cd = ["PROVINSI", "year", "growth_pdrb_hijau", "ln_pmtb_per_pekerja", "IPM", "ln_pdrb_kapita"]
data_cd = df[kolom_cd].dropna().set_index(["PROVINSI", "year"])
y_cd = data_cd["growth_pdrb_hijau"]
X_cd = data_cd.drop(columns=["growth_pdrb_hijau"])
fe_cd = PanelOLS(y_cd, X_cd, entity_effects=True, drop_absorbed=True).fit()
resid_wide = fe_cd.resids.reset_index().pivot(index="year", columns="PROVINSI", values="residual"
                                               if "residual" in fe_cd.resids.reset_index().columns else 0)
resid_long = fe_cd.resids.reset_index()
resid_long.columns = ["PROVINSI", "year", "resid"]
resid_wide = resid_long.pivot(index="year", columns="PROVINSI", values="resid")
provinsi_list = resid_wide.columns.tolist()
N_prov = len(provinsi_list)
numerator = 0.0
for i in range(N_prov):
    for j in range(i + 1, N_prov):
        pasangan = resid_wide[[provinsi_list[i], provinsi_list[j]]].dropna()
        if len(pasangan) < 3:
            continue
        rho_ij = pasangan.iloc[:, 0].corr(pasangan.iloc[:, 1])
        if not np.isnan(rho_ij):
            numerator += np.sqrt(len(pasangan)) * rho_ij
CD_stat = np.sqrt(2 / (N_prov * (N_prov - 1))) * numerator
p_cd = 2 * (1 - stats.norm.cdf(abs(CD_stat)))
print(f"CD statistic = {CD_stat:.4f}, p-value = {p_cd:.4f}")
CD_SIGNIFIKAN = p_cd < 0.05
if CD_SIGNIFIKAN:
    print("--> SIGNIFIKAN: ada cross-sectional dependence -- WAJIB masukkan dummy "
          "waktu di spesifikasi GMM (Tahap VI) sbg mitigasi parsial, dan sebut "
          "sbg keterbatasan (LLC/IPS generasi-1 & GMM standar mengasumsikan "
          "independensi antar-unit).")
else:
    print("--> TIDAK SIGNIFIKAN: asumsi independensi antar-provinsi cukup aman.")


TAHAP III -- UJI PRASYARAT PANEL
[III.1] Fisher/Maddala-Wu -- LEVEL ln(PDRB Hijau): P=41.0234 (df=68, N=34), p=0.996078 -> TIDAK STASIONER (dekat unit root)
[III.1] Fisher/Maddala-Wu -- PERTUMBUHAN Delta ln: P=254.6888 (df=68, N=34), p=0.000000 -> STASIONER

[III -- CATATAN WAJIB] Fisher/Maddala-Wu SAH secara metodologis (Maddala & Wu, 1999) dan justru LEBIH cocok utk panel unbalanced (Kalimantan Utara) drpd LLC yang butuh T seragam. TAPI penguji sering mengharapkan nama uji standar LLC/IPS -- kedua uji itu TIDAK tersedia matang di Python. WAJIB jalankan Tahap III di skrip R (levinLin, ips.test) sbg pelengkap/penguat, bukan pengganti.

[III.2] Pesaran CD test (cross-sectional dependence) -- spesifikasi growth:
CD statistic = 30.3899, p-value = 0.0000
--> SIGNIFIKAN: ada cross-sectional dependence -- WAJIB masukkan dummy waktu di spesifikasi GMM (Tahap VI) sbg mitigasi parsial, dan sebut sbg keterbatasan (LLC/IPS generasi-1 & GMM standar mengasumsikan independensi antar-unit).


In [ ]:
# #####################################################################
# TAHAP IV -- MODEL PANEL STATIS (BASELINE/PEMBANDING, TANPA LAG)
# #####################################################################
print(f"\n{'='*70}\nTAHAP IV -- MODEL PANEL STATIS (Chow -> LM -> Hausman)\n{'='*70}")
kolom_statis = ["PROVINSI", "year", "ln_pdrb_hijau", "ln_pmtb_per_pekerja", "IPM", "ln_pdrb_kapita"]
data_statis = df[kolom_statis].dropna().set_index(["PROVINSI", "year"])
y_s = data_statis["ln_pdrb_hijau"]
X_s = data_statis.drop(columns=["ln_pdrb_hijau"]).assign(const=1)

X_s_fe = X_s.drop(columns=["const"])

pooled_s = PooledOLS(y_s, X_s).fit()
fe_s = PanelOLS(y_s, X_s_fe, entity_effects=True, drop_absorbed=True).fit()
re_s = RandomEffects(y_s, X_s).fit()

f_pooled = fe_s.f_pooled
print(f"[IV.1] Chow (F-test): F={f_pooled.stat:.4f}, p={f_pooled.pval:.6f} -> "
      f"{'FE diperlukan' if f_pooled.pval < 0.05 else 'Pooled cukup'}")

resid_df = pooled_s.resids.reset_index()
resid_df.columns = ["PROVINSI", "year", "resid"]
N_, T_bar = resid_df["PROVINSI"].nunique(), len(resid_df) / resid_df["PROVINSI"].nunique()
A = ((resid_df.groupby("PROVINSI")["resid"].sum() ** 2) / resid_df.groupby("PROVINSI")["resid"].count()).sum()
B = (resid_df["resid"] ** 2).sum()
LM = (len(resid_df) / (2 * (T_bar - 1))) * ((A / B) - 1) ** 2
p_lm = 1 - stats.chi2.cdf(LM, df=1)
print(f"[IV.2] Breusch-Pagan LM: LM={LM:.4f}, p={p_lm:.6f} -> "
      f"{'RE diperlukan' if p_lm < 0.05 else 'Pooled cukup'}")

kolom_sama = [c for c in fe_s.params.index if c in re_s.params.index and c != "const"]
diff_h = fe_s.params[kolom_sama] - re_s.params[kolom_sama]
Var_diff = fe_s.cov.loc[kolom_sama, kolom_sama] - re_s.cov.loc[kolom_sama, kolom_sama]
try:
    hausman_stat = float(diff_h.values @ np.linalg.inv(Var_diff.values) @ diff_h.values)
    p_hausman = 1 - stats.chi2.cdf(hausman_stat, df=len(kolom_sama))
    print(f"[IV.3] Hausman: stat={hausman_stat:.4f}, p={p_hausman:.6f} -> "
          f"{'FE konsisten' if p_hausman < 0.05 else 'RE konsisten & efisien'}")
except np.linalg.LinAlgError:
    print("[IV.3] Hausman: matriks singular -- andalkan Chow & LM saja.")
print("[IV -- CATATAN] Hasil ini BASELINE/PEMBANDING (model level tanpa lag) -- "
      "BUKAN kandidat model final. Batas Pooled/FE dipakai ulang di Tahap V.")


TAHAP IV -- MODEL PANEL STATIS (Chow -> LM -> Hausman)
[IV.1] Chow (F-test): F=1435.0154, p=0.000000 -> FE diperlukan
[IV.2] Breusch-Pagan LM: LM=0.0107, p=0.917531 -> Pooled cukup
[IV.3] Hausman: stat=0.4268, p=0.934657 -> RE konsisten & efisien
[IV -- CATATAN] Hasil ini BASELINE/PEMBANDING (model level tanpa lag) -- BUKAN kandidat model final. Batas Pooled/FE dipakai ulang di Tahap V.


In [ ]:
# #####################################################################
# TAHAP V -- UJI KEBUTUHAN MODEL DINAMIS (BOND'S RULE OF THUMB)
# #####################################################################
print(f"\n{'='*70}\nTAHAP V -- BOND'S RULE OF THUMB (Bond, 2002)\n{'='*70}")


def hitung_batas_bond(df_input, kolom_dep):
    dfx = df_input.set_index(["PROVINSI", "year"]).copy()
    dfx["_lag"] = df_input.groupby("PROVINSI")[kolom_dep].shift(1).values
    valid = dfx.dropna(subset=[kolom_dep, "_lag"])
    p_ = PooledOLS.from_formula(f"{kolom_dep} ~ 1 + _lag", data=valid).fit()
    f_ = PanelOLS.from_formula(f"{kolom_dep} ~ 1 + _lag + EntityEffects", data=valid).fit()
    return sorted([f_.params["_lag"], p_.params["_lag"]])


kolom_gmm_growth = ["PROVINSI", "year", "growth_pdrb_hijau", "ln_pmtb_per_pekerja",
                     "IPM", "ln_pdrb_kapita", "ndvi_mean", "lst_mean"]
kolom_gmm_level = ["PROVINSI", "year", "ln_pdrb_hijau", "ln_pmtb_per_pekerja",
                    "IPM", "ln_pdrb_kapita", "ndvi_mean", "lst_mean"]
df_gmm_growth = df[kolom_gmm_growth].dropna()
df_gmm_level = df[kolom_gmm_level].dropna()

batas_bawah_growth, batas_atas_growth = hitung_batas_bond(df_gmm_growth, "growth_pdrb_hijau")
batas_bawah_level, batas_atas_level = hitung_batas_bond(df_gmm_level, "ln_pdrb_hijau")
print(f"[V.1] Rentang valid PERTUMBUHAN: [{batas_bawah_growth:.4f}, {batas_atas_growth:.4f}]")
print(f"[V.2] Rentang valid LEVEL      : [{batas_bawah_level:.4f}, {batas_atas_level:.4f}]")
print("[V -- CATATAN] rho_GMM WAJIB dicek terhadap rentang ini di Tahap VII -- "
      "kalau meleset tapi Hansen & AR(2) tetap lolos, itu BUKAN otomatis gagal "
      "(rentang bisa sangat sempit), tapi WAJIB didiskusikan eksplisit, bukan diam-diam.")


TAHAP V -- BOND'S RULE OF THUMB (Bond, 2002)
[V.1] Rentang valid PERTUMBUHAN: [-0.3795, -0.3298]
[V.2] Rentang valid LEVEL      : [0.9424, 0.9932]
[V -- CATATAN] rho_GMM WAJIB dicek terhadap rentang ini di Tahap VII -- kalau meleset tapi Hansen & AR(2) tetap lolos, itu BUKAN otomatis gagal (rentang bisa sangat sempit), tapi WAJIB didiskusikan eksplisit, bukan diam-diam.


In [ ]:
# #####################################################################
# TAHAP VI -- SPESIFIKASI & ESTIMASI MODEL DINAMIS (GMM)
# #####################################################################
print(f"\n{'='*70}\nTAHAP VI -- ESTIMASI GMM DINAMIS (Diff & System, Level & Growth)\n{'='*70}")

SPEC_LEVEL = "ln_pdrb_hijau L(1:1).ln_pdrb_hijau ln_pmtb_per_pekerja IPM ln_pdrb_kapita ndvi_mean lst_mean"
GMM_STYLE_LEVEL = "gmm(ln_pdrb_hijau, 2:4)"
SPEC_GROWTH = "growth_pdrb_hijau L(1:1).growth_pdrb_hijau ln_pmtb_per_pekerja IPM ln_pdrb_kapita ndvi_mean lst_mean"
GMM_STYLE_GROWTH = "gmm(growth_pdrb_hijau, 2:4)"
# CATATAN ASUMSI (lihat blok komentar atas skrip): semua regressor non-lag
# diperlakukan strictly exogenous (iv-style) -- sebutkan eksplisit di Bab III.
IV_STYLE = "iv(ln_pmtb_per_pekerja IPM ln_pdrb_kapita ndvi_mean lst_mean)"


def jalankan_gmm(df_input, spec, gmm_style, label, twostep=True):
    print(f"\n--- {label} ---")
    print(f"Observasi: {len(df_input)} | Provinsi: {df_input['PROVINSI'].nunique()}")
    opsi = "collapse"
    if not twostep: # If twostep is False, we explicitly ask for one-step
        opsi += " onestep"
    # If twostep is True, we rely on pydynpd's default behavior for two-step GMM,
    # or the fact that 'twostep' as a string option might be disallowed.

    hasil = {}
    for nama_model, opsi_tambahan in [("Difference GMM", "nolevel fod"), ("System GMM", "fod")]:
        current_opsi = f"{opsi} {opsi_tambahan}"
        cmd = f"{spec} | {gmm_style} {IV_STYLE} | {current_opsi}"
        m = regression.abond(cmd, df_input, ["PROVINSI", "year"])
        hasil[nama_model] = m.models[0]
    return hasil


hasil_level = jalankan_gmm(df_gmm_level, SPEC_LEVEL, GMM_STYLE_LEVEL, "LEVEL (pembanding)")
hasil_growth = jalankan_gmm(df_gmm_growth, SPEC_GROWTH, GMM_STYLE_GROWTH, "PERTUMBUHAN (kandidat utama)")


TAHAP VI -- ESTIMASI GMM DINAMIS (Diff & System, Level & Growth)

--- LEVEL (pembanding) ---
Observasi: 538 | Provinsi: 34
 Dynamic panel-data estimation, two-step difference GMM
 Group variable: PROVINSI                        Number of obs = 468      
 Time variable: year                             Min obs per group: 9     
 Number of instruments = 8                       Max obs per group: 14    
 Number of groups = 34                           Avg obs per group: 13.76 
+---------------------+------------+---------------------+------------+-----------+---+
|    ln_pdrb_hijau    |   coef.    | Corrected Std. Err. |     z      |   P>|z|   |   |
+---------------------+------------+---------------------+------------+-----------+---+
|   L1.ln_pdrb_hijau  | 0.3970963  |      0.5804855      | 0.6840763  | 0.4939270 |   |
| ln_pmtb_per_pekerja | 0.2890585  |      0.2204767      | 1.3110612  | 0.1898371 |   |
|         IPM         | 0.0238019  |      0.0339767      | 0.7005371  | 0.483591

In [ ]:
# #####################################################################
# TAHAP VII -- DIAGNOSTIK (jumlah instrumen & cek Windmeijer)
# #####################################################################
print(f"\n{'='*70}\nTAHAP VII -- DIAGNOSTIK LENGKAP\n{'='*70}")


def ekstrak_diagnostik(model_obj, batas_bawah, batas_atas, jumlah_grup):
    tabel = model_obj.regression_table
    baris_lag = tabel[tabel["variable"].str.contains(r"L1\.", regex=True)]
    koef = baris_lag["coefficient"].values[0]
    p_koef = baris_lag["p_value"].values[0]
    hansen_p = model_obj.hansen.p_value
    ar1_p = model_obj.AR_list[0].P_value
    ar2_p = model_obj.AR_list[1].P_value

    num_indep = len(tabel)
    num_instru = model_obj.hansen.df + num_indep
    rasio_instru_grup = num_instru / jumlah_grup

    return {
        "koefisien": koef, "p_koefisien": p_koef,
        "hansen_p": hansen_p, "ar1_p": ar1_p, "ar2_p": ar2_p,
        "lolos_bond": batas_bawah <= koef <= batas_atas,
        "masuk_akal": abs(koef) < 1,
        "jumlah_instrumen": int(round(num_instru)),
        "jumlah_grup": jumlah_grup,
        "rasio_instru_grup": rasio_instru_grup,
        "instrumen_aman": num_instru <= jumlah_grup,
    }


JUMLAH_GRUP = df["PROVINSI"].nunique()

diag = {}
for spek, hasil, (bb, ba) in [("Level", hasil_level, (batas_bawah_level, batas_atas_level)),
                               ("Growth", hasil_growth, (batas_bawah_growth, batas_atas_growth))]:
    for estimator in ["Difference GMM", "System GMM"]:
        d = ekstrak_diagnostik(hasil[estimator], bb, ba, JUMLAH_GRUP)
        diag[f"{spek}-{estimator}"] = d
        print(f"\n[{spek} / {estimator}]")
        print(f"  Koef lag = {d['koefisien']:.4f} (p={d['p_koefisien']:.4f}) | "
              f"Bond OK? {d['lolos_bond']} | |koef|<1? {d['masuk_akal']}")
        print(f"  Hansen p = {d['hansen_p']:.4f} {'[LOLOS]' if d['hansen_p']>0.05 else '[GAGAL]'} | "
              f"AR1 p={d['ar1_p']:.4f} | AR2 p={d['ar2_p']:.4f} {'[LOLOS]' if d['ar2_p']>0.05 else '[GAGAL]'}")
        print(f"  Jumlah instrumen = {d['jumlah_instrumen']} vs jumlah grup = {d['jumlah_grup']} "
              f"-> {'AMAN' if d['instrumen_aman'] else '[PERINGATAN] INSTRUMEN > GRUP -- overfitting risk (Roodman, 2009)'}")


TAHAP VII -- DIAGNOSTIK LENGKAP

[Level / Difference GMM]
  Koef lag = 0.3971 (p=0.4939) | Bond OK? False | |koef|<1? True
  Hansen p = 0.0141 [GAGAL] | AR1 p=0.2773 | AR2 p=0.2902 [LOLOS]
  Jumlah instrumen = 8 vs jumlah grup = 34 -> AMAN

[Level / System GMM]
  Koef lag = 1.1688 (p=0.0000) | Bond OK? False | |koef|<1? False
  Hansen p = 0.5069 [LOLOS] | AR1 p=0.0862 | AR2 p=0.1779 [LOLOS]
  Jumlah instrumen = 10 vs jumlah grup = 34 -> AMAN

[Growth / Difference GMM]
  Koef lag = -0.2749 (p=0.0000) | Bond OK? False | |koef|<1? True
  Hansen p = 0.2452 [LOLOS] | AR1 p=0.0625 | AR2 p=0.2300 [LOLOS]
  Jumlah instrumen = 8 vs jumlah grup = 34 -> AMAN

[Growth / System GMM]
  Koef lag = -0.2943 (p=0.0005) | Bond OK? False | |koef|<1? True
  Hansen p = 0.2692 [LOLOS] | AR1 p=0.0523 | AR2 p=0.2790 [LOLOS]
  Jumlah instrumen = 10 vs jumlah grup = 34 -> AMAN


In [ ]:
# VII.2 -- Verifikasi Windmeijer: bandingkan SE one-step vs two-step.
print(f"\n{'='*70}\n[VII.2] Verifikasi kasar Windmeijer -- one-step vs two-step SE\n{'='*70}")
hasil_growth_onestep = jalankan_gmm(df_gmm_growth, SPEC_GROWTH, GMM_STYLE_GROWTH,
                                     "PERTUMBUHAN (one-step, utk banding SE)", twostep=False)
for estimator in ["Difference GMM", "System GMM"]:
    tabel_1s = hasil_growth_onestep[estimator].regression_table
    tabel_2s = hasil_growth[estimator].regression_table
    baris_1s = tabel_1s[tabel_1s["variable"].str.contains(r"L1\.")]
    baris_2s = tabel_2s[tabel_2s["variable"].str.contains(r"L1\.")]
    se_1s = baris_1s["std_err"].values[0] if "std_err" in tabel_1s.columns else np.nan
    se_2s = baris_2s["std_err"].values[0] if "std_err" in tabel_2s.columns else np.nan
    print(f"  {estimator}: SE one-step={se_1s:.4f} vs SE two-step={se_2s:.4f} "
          f"(rasio={se_2s/se_1s if se_1s else np.nan:.2f}) "
          f"-- kalau rasio << 1 secara drastis TANPA penjelasan, curigai SE two-step "
          f"belum dikoreksi Windmeijer; cross-check WAJIB via R.")
print("[VII -- CATATAN NAMA KOLOM] Sesuaikan 'std_err' dgn nama kolom aktual di "
      "regression_table Anda (print(tabel.columns) dulu bila error) -- penamaan "
      "kolom bisa berbeda antar versi pydynpd.")


[VII.2] Verifikasi kasar Windmeijer -- one-step vs two-step SE

--- PERTUMBUHAN (one-step, utk banding SE) ---
Observasi: 504 | Provinsi: 34
 Dynamic panel-data estimation, one-step difference GMM
 Group variable: PROVINSI                        Number of obs = 435      
 Time variable: year                             Min obs per group: 9     
 Number of instruments = 8                       Max obs per group: 13    
 Number of groups = 34                           Avg obs per group: 12.79 
+----------------------+------------+---------------------+------------+-----------+-----+
|  growth_pdrb_hijau   |   coef.    | Corrected Std. Err. |     z      |   P>|z|   |     |
+----------------------+------------+---------------------+------------+-----------+-----+
| L1.growth_pdrb_hijau | -0.3751706 |      0.1016387      | -3.6912173 | 0.0002232 | *** |
| ln_pmtb_per_pekerja  | 0.1267381  |      0.0499676      | 2.5364043  | 0.0111997 |  *  |
|         IPM          | -0.0082977 |      0.00

In [ ]:
# #####################################################################
# TAHAP VIII -- ROBUSTNESS
# #####################################################################
print(f"\n{'='*70}\nTAHAP VIII -- ROBUSTNESS\n{'='*70}")

df_tanpa_kaltara = df[df["PROVINSI"] != ENTITAS_KHUSUS]
df_gmm_growth_tanpa = df_tanpa_kaltara[kolom_gmm_growth].dropna()
hasil_tanpa_kaltara = jalankan_gmm(df_gmm_growth_tanpa, SPEC_GROWTH, GMM_STYLE_GROWTH,
                                    "TANPA KALIMANTAN UTARA (growth)")
JUMLAH_GRUP_TANPA = df_tanpa_kaltara["PROVINSI"].nunique()
for estimator in ["Difference GMM", "System GMM"]:
    d = ekstrak_diagnostik(hasil_tanpa_kaltara[estimator], batas_bawah_growth, batas_atas_growth, JUMLAH_GRUP_TANPA)
    print(f"[Tanpa Kaltara / {estimator}] Hansen p={d['hansen_p']:.4f}, AR2 p={d['ar2_p']:.4f}, "
          f"instrumen {d['jumlah_instrumen']}/{d['jumlah_grup']}")
    diag[f"Growth-TanpaKaltara-{estimator}"] = d

print("\n[VIII.2] Robustness -- 3 skenario valuasi PDRB Hijau:")
skenario_kolom = {"Utama (SCC 3%)": "pdrb_hijau_utama",
                  "Sensitivitas DALY x1": "pdrb_hijau_sensitivitas_daly_1x",
                  "Sensitivitas DALY x3": "pdrb_hijau_sensitivitas_daly_3x"}
hasil_skenario_valuasi = {}
for nama_sk, kolom_dep in skenario_kolom.items():
    if kolom_dep not in df.columns:
        print(f"  {nama_sk}: kolom '{kolom_dep}' tidak ditemukan -- lewati (cek nama kolom Anda).")
        continue
    df[f"ln_{kolom_dep}"] = np.log(df[kolom_dep].where(df[kolom_dep] > 0))
    df[f"growth_{kolom_dep}"] = df.groupby("PROVINSI")[f"ln_{kolom_dep}"].diff()
    kolom_sk = ["PROVINSI", "year", f"growth_{kolom_dep}", "ln_pmtb_per_pekerja", "IPM",
                "ln_pdrb_kapita", "ndvi_mean", "lst_mean"]
    df_sk = df[kolom_sk].dropna()
    spec_sk = SPEC_GROWTH.replace("growth_pdrb_hijau", f"growth_{kolom_dep}")
    gmm_sk = GMM_STYLE_GROWTH.replace("growth_pdrb_hijau", f"growth_{kolom_dep}")
    try:
        m_sk = regression.abond(f"{spec_sk} | {gmm_sk} {IV_STYLE} | collapse nolevel fod",
                                 df_sk, ["PROVINSI", "year"])
        model_sk = m_sk.models[0]
        d_sk = ekstrak_diagnostik(model_sk, batas_bawah_growth, batas_atas_growth, df_sk["PROVINSI"].nunique())
        hasil_skenario_valuasi[nama_sk] = d_sk
        print(f"  {nama_sk}: koef={d_sk['koefisien']:.4f}, Hansen p={d_sk['hansen_p']:.4f}, "
              f"AR2 p={d_sk['ar2_p']:.4f}, instrumen {d_sk['jumlah_instrumen']}/{d_sk['jumlah_grup']}")
        diag[f"Growth-{nama_sk}"] = d_sk
    except Exception as e:
        print(f"  {nama_sk}: GAGAL diestimasi ({e})")

print("\n[VIII -- CATATAN WAJIB] JANGAN ada sel kosong pada rekap skenario di atas "
      "sebelum ke Tahap IX -- kalau ada yang gagal, investigasi dulu (biasanya kolom "
      "hilang atau observasi terlalu sedikit setelah dropna).")


TAHAP VIII -- ROBUSTNESS

--- TANPA KALIMANTAN UTARA (growth) ---
Observasi: 493 | Provinsi: 33
 Dynamic panel-data estimation, two-step difference GMM
 Group variable: PROVINSI                        Number of obs = 426      
 Time variable: year                             Min obs per group: 10    
 Number of instruments = 8                       Max obs per group: 13    
 Number of groups = 33                           Avg obs per group: 12.91 
+----------------------+------------+---------------------+------------+-----------+-----+
|  growth_pdrb_hijau   |   coef.    | Corrected Std. Err. |     z      |   P>|z|   |     |
+----------------------+------------+---------------------+------------+-----------+-----+
| L1.growth_pdrb_hijau | -0.2812059 |      0.0678194      | -4.1463941 | 0.0000338 | *** |
| ln_pmtb_per_pekerja  | 0.0810495  |      0.0608962      | 1.3309453  | 0.1832070 |     |
|         IPM          | -0.0079211 |      0.0032201      | -2.4599068 | 0.0138973 |  *  |
|

In [ ]:
# #####################################################################
# TAHAP IX -- EKSPOR
# #####################################################################
print(f"\n{'='*70}\nTAHAP X -- EKSPOR LAMPIRAN\n{'='*70}")
rekap_lampiran = pd.DataFrame(diag).T
rekap_lampiran.to_csv(f"{FOLDER_OUTPUT}/lampiran_diagnostik_gmm_lengkap.csv")
print(f"[X] Rekap diagnostik SEMUA spesifikasi & estimator -> "
      f"{FOLDER_OUTPUT}/lampiran_diagnostik_gmm_lengkap.csv")
print(rekap_lampiran[["koefisien", "hansen_p", "ar2_p", "jumlah_instrumen", "jumlah_grup", "instrumen_aman"]]
      .to_string())


TAHAP X -- EKSPOR LAMPIRAN
[X] Rekap diagnostik SEMUA spesifikasi & estimator -> /content/drive/MyDrive/DatabaseSkripsi/Outputs/lampiran_diagnostik_gmm_lengkap.csv
                                   koefisien  hansen_p     ar2_p jumlah_instrumen jumlah_grup instrumen_aman
Level-Difference GMM                0.397096   0.01406  0.290213                8          34           True
Level-System GMM                     1.16883  0.506869   0.17787               10          34           True
Growth-Difference GMM              -0.274918  0.245156  0.230009                8          34           True
Growth-System GMM                  -0.294289  0.269236   0.27898               10          34           True
Growth-TanpaKaltara-Difference GMM -0.281206  0.224771  0.255025                8          33           True
Growth-TanpaKaltara-System GMM     -0.299598  0.261226  0.299675               10          33           True
Growth-Utama (SCC 3%)              -0.274918  0.245156  0.230009        